In [62]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col , udf
from pyspark.sql.types import StringType , IntegerType
import time

In [63]:
spark = SparkSession.builder.appName("spark-demo-app").getOrCreate()

In [64]:
df_data = spark.range(0,1000).withColumn("value" , col("id")%1000)

In [65]:
df_data.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [66]:
print("Before Partition :" , df_data.rdd.getNumPartitions())

Before Partition : 12


In [67]:
df_repartitioned = df_data.repartition(20)

In [68]:
print("After Partition:" , df_repartitioned.rdd.getNumPartitions())

After Partition: 20


In [69]:
df_coalesced = df_repartitioned.coalesce(12)

In [70]:
print("After coalesced:" , df_coalesced.rdd.getNumPartitions())

After coalesced: 12


In [71]:
df_coalesced2 = df_repartitioned.coalesce(2)

In [72]:
print("After coalesced:" , df_coalesced2.rdd.getNumPartitions())

After coalesced: 2


In [73]:
df_repartitioned.write.mode("overwrite").csv("output/employee_data_afterPartitioned",header=True)

In [74]:
df_coalesced.write.mode("overwrite").csv("output/employee_data_afterCoalesced" , header=True)

In [75]:
#optimization and chaching

In [76]:
optimized_df= df_data.filter(col("value")>500).filter(col("id")<800).select("id" , "value")

In [77]:
optimized_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [id#166L, value#168L]
      +- InMemoryRelation [id#166L, value#168L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) Project [id#15L, (id#15L % 1000) AS value#17L]
               +- *(1) Filter (((id#15L % 1000) > 500) AND (id#15L < 800))
                  +- *(1) Range (0, 1000, step=1, splits=12)




In [78]:
start_time = time.time()
count_uncached= optimized_df.count() 
end_time = time.time()
print(f"1. Optimized Execution | Count: {count_uncached} | Time:{round(end_time - start_time ,4)} seconds")

1. Optimized Execution | Count: 299 | Time:0.1367 seconds


In [79]:
optimized_df.cache()

26/06/13 06:44:57 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [80]:
start_time = time.time()
count_first_cached= optimized_df.count() 
end_time = time.time()
print(f"2. First Cache Action | Count: {count_first_cached} | Time:{round(end_time - start_time ,4)} seconds")

2. First Cache Action | Count: 299 | Time:0.169 seconds


In [81]:
#udfs

In [88]:
data=[("Alice" , 25), ("Bob", 17) , ("Charlie" ,35)]
columns= ["name" ,"Age"]
df1= spark.createDataFrame(data , columns)

In [89]:
df1.show()

+-------+---+
|   name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 35|
+-------+---+



In [90]:
def categorize_age(age):
    if age>=18:
        return "Adult"
    return "Minor"

In [91]:
age_category_udf = udf(categorize_age , StringType())

In [94]:
df_method1 = df1.withColumn("Category" , age_category_udf(col("Age")))
print("method 1: Standard UDF via DataFrame API")
df_method1.show()

method 1: Standard UDF via DataFrame API


+-------+---+--------+
|   name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
+-------+---+--------+



In [98]:
def categorize_drive(age):
    if age >= 18:
        return "Can Drive"
    elif age>=13:
        return "can drive but with learning liscence"
    else:
        return "can't drive"

In [99]:
drive_category_udf = udf(categorize_drive , StringType())

In [100]:
df_method2= df1.withColumn("category", drive_category_udf(col("Age")))
df_method2.show()

+-------+---+--------------------+
|   name|Age|            category|
+-------+---+--------------------+
|  Alice| 25|           Can Drive|
|    Bob| 17|can drive but wit...|
|Charlie| 35|           Can Drive|
+-------+---+--------------------+



In [ ]:
# for sql

In [101]:
spark.udf.register("sql_categorize_age" , categorize_age , StringType())
df1.createOrReplaceTempView("people")

In [102]:
sql_df = spark.sql("""
    SELECT 
        name,
        Age,
        sql_categorize_age(Age) AS Category
    FROM people
""")

sql_df.show()

+-------+---+--------+
|   name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
+-------+---+--------+

